# Capítulo 7 · Laboratório 7: as ondas de Elliott em Python
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap07.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Um checador que aplica as três regras de Elliott a qualquer contagem, um caçador de impulsos no Ibovespa e no Dow que compara o mercado real com mercados de mentira (os mesmos retornos em ordem sorteada), as diretrizes de Fibonacci e de alternância, o oscilador de Elliott, o teste do tempo da contagem de Steve Hochberg e o índice dos arranha-céus.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap07.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
SERIES = {"^BVSP": "1993-01-01", "^DJI": "1992-01-01", "^GSPC": "1927-01-01", "GC=F": "2000-01-01",
          "PETR4.SA": "2010-01-01", "EURUSD=X": "2004-01-01"}             # euro só para as figuras


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=False, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for t, ini in SERIES.items():
    if t not in D or D[t].index[0] > pd.Timestamp(ini) + pd.Timedelta(days=400):
        D[t] = baixa(t, ini)
confere({t: D[t] for t in SERIES})
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    return d[ok].copy()


IBOV, DOW, SPX, OURO, PETR = (limpa(D[t]) for t in ("^BVSP", "^DJI", "^GSPC", "GC=F", "PETR4.SA"))
IBOV = IBOV.loc["1995":]
print(f"Parte 1. Ibovespa desde {IBOV.index[0]:%Y}, Dow desde {DOW.index[0]:%Y}, S&P 500 desde {SPX.index[0]:%Y},"
      f" ouro desde {OURO.index[0]:%Y}; último pregão {IBOV.index[-1]:%d/%m/%Y}.")
rec = IBOV.loc["2008", "Close"]
print(f"   Recorde de fechamento do Ibovespa em 2008: {nr(rec.max(), 0)} em {rec.idxmax():%d/%m/%Y}")

## Parte 2: o checador de contagem

In [ ]:
def checar_contagem(p):
    """p = [início da onda 1, fim da 1, fim da 2, fim da 3, fim da 4, fim da 5] (alta ou baixa)."""
    s = 1 if p[1] > p[0] else -1                      # alta: +1, baixa: -1
    q = [s * x for x in p]                            # a baixa vira alta para usar as mesmas contas
    w1, w3, w5 = q[1] - q[0], q[3] - q[2], q[5] - q[4]
    problemas = []
    if q[2] <= q[0]:
        problemas.append("regra 1: a onda 2 passou do início da onda 1")
    if w3 < w1 and w3 < w5:
        problemas.append("regra 2: a onda 3 é a mais curta")
    if q[4] <= q[1]:
        problemas.append("regra 3: a onda 4 invadiu o território da onda 1")
    razoes = {"onda 2 / onda 1": (q[1] - q[2]) / w1, "onda 3 / onda 1": w3 / w1,
              "onda 4 / onda 3": (q[3] - q[4]) / w3, "onda 5 / onda 1": w5 / w1}
    return problemas, {k: round(float(v), 3) for k, v in razoes.items()}


print("\nParte 2. Checador de contagem")
for nome, p in (("contagem válida", [100, 120, 110, 150, 135, 160]),
                ("onda 4 dentro da onda 1", [100, 120, 108, 150, 118, 160]),
                ("onda 3 mais curta", [100, 130, 115, 135, 128, 160])):
    prob, raz = checar_contagem(p)
    print(f"   {nome}: {p} -> {prob or 'passa nas três regras'}; razões {raz}")

In [ ]:
# Sua vez: digite os seis pontos de uma contagem (início da onda 1 e o fim das ondas 1 a 5)
minha = [100, 125, 112, 160, 140, 175]
print(checar_contagem(minha))

## Parte 3: o ziguezague

In [ ]:
def zigzag(precos, limiar=0.05):
    """Pivôs pelo fechamento: [(posição, preço)], alternando topo e fundo. Um topo (fundo) se confirma
    quando o preço cai (sobe) 'limiar' a partir dele."""
    p = np.asarray(precos, dtype=float)
    piv, tend, hi, lo, ext = [], 0, 0, 0, 0
    for i in range(1, len(p)):
        if tend == 0:
            hi = i if p[i] > p[hi] else hi
            lo = i if p[i] < p[lo] else lo
            if p[i] >= p[lo] * (1 + limiar):
                piv.append((lo, p[lo]))
                tend, ext = 1, i
            elif p[i] <= p[hi] * (1 - limiar):
                piv.append((hi, p[hi]))
                tend, ext = -1, i
        elif tend == 1:
            if p[i] > p[ext]:
                ext = i
            elif p[i] <= p[ext] * (1 - limiar):
                piv.append((ext, p[ext]))
                tend, ext = -1, i
        else:
            if p[i] < p[ext]:
                ext = i
            elif p[i] >= p[ext] * (1 + limiar):
                piv.append((ext, p[ext]))
                tend, ext = 1, i
    return piv

## Parte 4: o caçador de impulsos

In [ ]:
def impulsos(piv):
    """Toda janela de 6 pivôs que anda para a frente (3 passa de 1, 5 passa de 3) é candidata;
    devolve as que passam nas três regras e o total de candidatas."""
    validos, total = [], 0
    for i in range(len(piv) - 5):
        pos_ = [x[0] for x in piv[i:i + 6]]
        p = [x[1] for x in piv[i:i + 6]]
        s = 1 if p[1] > p[0] else -1
        q = [s * x for x in p]
        if not (q[3] > q[1] and q[5] > q[3]):
            continue
        total += 1
        problemas, razoes = checar_contagem(p)
        if not problemas:
            razoes["pos"] = pos_
            razoes["sentido"] = s
            validos.append(razoes)
    return validos, total


perto = lambda x, alvo, tol: abs(x / alvo - 1) <= tol


def resumo(validos, total):
    r = {"candidatos": total, "passam": len(validos), "% que passa": len(validos) / total if total else np.nan}
    if validos:
        r3 = np.array([v["onda 3 / onda 1"] for v in validos])
        r5 = np.array([v["onda 5 / onda 1"] for v in validos])
        r2 = np.array([v["onda 2 / onda 1"] for v in validos])
        r4 = np.array([v["onda 4 / onda 3"] for v in validos])
        r["onda 3 = 1,618 x onda 1 (±5%)"] = np.mean([perto(x, 1.618, 0.05) for x in r3])
        r["onda 5 = onda 1 (±10%)"] = np.mean([perto(x, 1.0, 0.10) for x in r5])
        r["onda 2 entre 50% e 61,8%"] = np.mean((r2 >= 0.5) & (r2 <= 0.618))
        r["onda 4 entre 23,6% e 38,2%"] = np.mean((r4 >= 0.236) & (r4 <= 0.382))
        r["onda 3 a maior"] = np.mean([v["onda 3 / onda 1"] >= max(1, v["onda 5 / onda 1"]) for v in validos])
    return r


def mercado_de_mentira(fech, n=100, semente=7, limiar=0.05):
    """Mesmos retornos diários, em ordem sorteada: sem memória nenhuma, por construção."""
    rng = np.random.default_rng(semente)
    ret = np.diff(np.log(fech))
    lista, soma = [], 0
    for _ in range(n):
        falso = fech[0] * np.exp(np.concatenate([[0], np.cumsum(rng.permutation(ret))]))
        v, t = impulsos(zigzag(falso, limiar))
        lista += v
        soma += t
    return resumo(lista, soma), lista


TAB, VAL = {}, {}
for nome, d in (("Ibovespa", IBOV), ("Dow Jones", DOW)):
    fech = d["Close"].to_numpy()
    v, t = impulsos(zigzag(fech))
    VAL[nome] = v
    TAB[f"{nome} real"] = resumo(v, t)
    TAB[f"{nome} embaralhado (100 vezes)"], VAL[f"{nome} falso"] = mercado_de_mentira(fech)
T = pd.DataFrame(TAB).T
print("\nParte 4. Caçador de impulsos (ziguezague de 5% no fechamento)")
fmt = T.copy()
for c in fmt.columns:
    fmt[c] = [nr(x, 0) if c in ("candidatos", "passam") else pc(x) for x in T[c]]
print(fmt.to_string())

In [ ]:
import matplotlib.pyplot as plt

def mostra_impulso(d, v, titulo):
    """Fechamento com os seis pivôs de um impulso que passou nas três regras."""
    i0, i5 = v["pos"][0], v["pos"][5]
    w = d.iloc[max(i0 - 20, 0):i5 + 40]
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(w.index, w["Close"], color="black", lw=0.9)
    for k, i in enumerate(v["pos"]):
        ax.annotate(str(k), (d.index[i], d["Close"].iloc[i]), fontsize=10, fontweight="bold")
    ax.set_title(titulo)
    plt.show()

mostra_impulso(IBOV, VAL["Ibovespa"][-1], "Ibovespa: o impulso mais recente que passou nas três regras")

## Parte 5: diretrizes

In [ ]:
def alternancia(validos, fech):
    """Alternância simples: a onda 2 é funda e rápida e a onda 4 rasa e lenta (ou o contrário)?"""
    alt = 0
    for v in validos:
        i0, i1, i2, i3, i4, i5 = v["pos"]
        prof2, prof4 = v["onda 2 / onda 1"], v["onda 4 / onda 3"]
        dur2 = (i2 - i1) / max(i1 - i0, 1)
        dur4 = (i4 - i3) / max(i3 - i2, 1)
        alt += (prof2 > prof4 and dur4 > dur2) or (prof4 > prof2 and dur2 > dur4)
    return alt / len(validos) if validos else np.nan


print("\nParte 5. Alternância (uma correção funda e curta, a outra rasa e longa):")
for nome, d in (("Ibovespa", IBOV), ("Dow Jones", DOW)):
    print(f"   {nome}: real {pc(alternancia(VAL[nome], d))}; embaralhado {pc(alternancia(VAL[nome + ' falso'], d))}")

## Parte 6: o oscilador de Elliott

In [ ]:
def ewo(fech):
    s = pd.Series(fech)
    return (s.rolling(5).mean() - s.rolling(35).mean()).to_numpy()


def onda3_pico(validos, fech):
    """Em quantos impulsos o maior valor do EWO (no sentido do impulso) cai dentro da onda 3?"""
    o = ewo(fech)
    ok = n = 0
    for v in validos:
        i0, i1, i2, i3, i4, i5 = v["pos"]
        if i0 < 35:
            continue
        seg = v["sentido"] * o[i0:i5 + 1]
        k = i0 + int(np.nanargmax(seg))
        n += 1
        ok += i2 <= k <= i3
    return ok / n if n else np.nan, n


print("\nParte 6. EWO (média de 5 menos média de 35): o maior pico cai na onda 3?")
for nome, d in (("Ibovespa", IBOV), ("Dow Jones", DOW)):
    fech = d["Close"].to_numpy()
    real, n = onda3_pico(VAL[nome], fech)
    rng = np.random.default_rng(7)
    ret = np.diff(np.log(fech))
    falsos = []
    for _ in range(30):
        f = fech[0] * np.exp(np.concatenate([[0], np.cumsum(rng.permutation(ret))]))
        falsos.append(onda3_pico(impulsos(zigzag(f))[0], f)[0])
    print(f"   {nome}: real {pc(real)} de {n} impulsos; mercados de mentira {pc(np.nanmean(falsos))}")
    duracao = np.mean([(v["pos"][3] - v["pos"][2]) / (v["pos"][5] - v["pos"][0]) for v in VAL[nome]])
    print(f"      (a onda 3 ocupa em média {pc(duracao)} do tempo do impulso)")

## Parte 7: o teste do tempo

In [ ]:
INVALIDA = 48133.54                                   # topo de 05/12/2025, fim da onda 5 na contagem de Hochberg
dw = DOW.loc["2025-12-06":]
acima_max = dw[dw["High"] > INVALIDA].index[0]
acima_fech = dw[dw["Close"] > INVALIDA].index[0]
ATE = (DOW.index.get_loc(acima_fech) - DOW.index.get_loc(pd.Timestamp("2025-12-05")))
MAX_DOW = dw["High"].max(), dw["High"].idxmax()
print(f"\nParte 7. Teste do tempo: nível de invalidação {nr(int(INVALIDA), 0)} (Dow, 05/12/2025)")
print(f"   máxima passou do nível em {acima_max:%d/%m/%Y}; primeiro fechamento acima em {acima_fech:%d/%m/%Y}"
      f" ({ATE} pregões depois); maior máxima desde então {nr(MAX_DOW[0], 0)} em {MAX_DOW[1]:%d/%m/%Y}")
o = OURO.loc["2025-10-01":]
print(f"   Ouro (futuro): máxima de {nr(o['High'].max(), 0)} dólares em {o['High'].idxmax():%d/%m/%Y};"
      f" fechamento de 30/09/2026: {nr(o['Close'].iloc[-1], 0)}")

## Parte 8: o índice dos arranha-céus

In [ ]:
PREDIOS = [("Chrysler", 1930), ("Empire State", 1931), ("World Trade Center", 1973), ("Sears Tower", 1974),
           ("Petronas", 1998), ("Taipei 101", 2004), ("Burj Khalifa", 2010), ("Merdeka 118", 2023)]
s = SPX["Close"]
print("\nParte 8. S&P 500 nos dois anos depois do recorde de altura (fim do ano de conclusão até dois anos depois):")
RET = []
for nome, ano in PREDIOS:
    a, b = s.loc[:f"{ano}-12-31"].iloc[-1], s.loc[:f"{ano + 2}-12-31"].iloc[-1]
    pior = s.loc[f"{ano + 1}":f"{ano + 2}"].min() / a - 1
    RET.append(b / a - 1)
    print(f"   {nome} ({ano}): {pc(b / a - 1)}; pior ponto {pc(pior)}")
dois = (s.resample("YE").last().pct_change(2).dropna())
print(f"   média de dois anos de qualquer período desde 1929: {pc(dois.loc['1931':].mean())};"
      f" depois dos prédios: {pc(np.mean(RET))} (8 casos)")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `impulsos` e `mercado_de_mentira` da Parte 4. Mostre como uma janela de seis pivôs vira candidata a impulso e por que embaralhar os retornos diários cria um mercado sem memória." Confira com a Parte 4, que imprime a Tabela 7.4: no Ibovespa real, 30 dos 106 candidatos passam nas três regras (28%), contra 23% nos mercados de mentira.
2. **Adaptar:** "Na célula 'Sua vez', logo depois da Parte 2, troque a lista `minha` pelos seis pontos de uma contagem minha em [ATIVO]: o início da onda 1 e o fim das ondas 1 a 5. Não mude a função `checar_contagem`." Antes de rodar a sua contagem, confira se a Parte 2 continua dizendo que a contagem `[100, 120, 108, 150, 118, 160]` invadiu o território da onda 1, como na Pergunta 3 do capítulo.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 4, troque o ziguezague de 5% para 8%. A diferença entre real e embaralhado muda?
2. Use o checador com uma contagem sua, de um gráfico que você acompanha.
3. Na Parte 6, troque o EWO de 5-35 por 5-34. O resultado muda?
4. Na Parte 8, inclua a Jeddah Tower em 2028 quando chegar a hora.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.